In [ ]:
# 1. Mount drive (optional for saves)
from google.colab import drive
drive.mount('/content/drive')

# 2. Install dependencies (if needed)
!pip install torch torchvision tqdm

# 3. Clone dataset from GitHub
!git clone https://github.com/IvonaTau/ikea.git
# Directory now contains 'images/', 'text_data/', etc.

# 4. Check image files
!find ikea/images -maxdepth 2 -type f | head -n 10
# You’ll see object images and maybe context scenes.

# 5. Prepare the image folder for ImageFolder
# We'll copy all images to one folder, or maintain subfolders by type.

import os
import shutil

src = 'ikea/images'
dst = 'data/images_all'
os.makedirs(dst, exist_ok=True)
for root, _, files in os.walk(src):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy(os.path.join(root, f), dst)

# 6. Set up training code (same as earlier script but as inline notebook)

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

# Hyperparams
latent_dim = 100
image_size = 64
batch_size = 128
epochs = 25
lr = 2e-4
beta1 = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Dataset & Dataloader
# The dataset is expected to be in a subdirectory of 'data' for ImageFolder.
# We need to make sure the 'images_all' folder is inside 'data'.
# The previous step already created 'data/images_all', so this should work.
dataset = datasets.ImageFolder('data', transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# Model definitions (Generator & Discriminator, same as script)

class Generator(nn.Module):
    def __init__(self, latent_dim, ngf=64, channels=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        if z.dim() == 2: z = z.unsqueeze(-1).unsqueeze(-1)
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, ndf=64, channels=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False)
        )

    def forward(self, img):
        return self.net(img).view(-1)

# Instantiate models
netG = Generator(latent_dim).to(device)
netD = Discriminator().to(device)
netG.apply(lambda m: nn.init.normal_(m.weight.data, 0.0, 0.02) if hasattr(m, 'weight') else None)
netD.apply(lambda m: nn.init.normal_(m.weight.data, 0.0, 0.02) if hasattr(m, 'weight') else None)

# Loss and optimizers
criterion = nn.BCEWithLogitsLoss()
optD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

fixed_noise = torch.randn(64, latent_dim, device=device)
real_label = 0.9  # label smoothing
fake_label = 0.0

# Training loop with live sample display
from IPython.display import clear_output, Image, display

for epoch in range(1, epochs + 1):
    for real, _ in tqdm(dataloader, desc=f"Epoch {epoch}/{epochs}"):
        real = real.to(device)
        b_size = real.size(0)
        netD.zero_grad()
        lab_real = torch.full((b_size,), real_label, device=device)
        lab_fake = torch.full((b_size,), fake_label, device=device)
        out_real = netD(real)
        lossD_real = criterion(out_real, lab_real)
        fake = netG(torch.randn(b_size, latent_dim, device=device))
        out_fake = netD(fake.detach())
        lossD_fake = criterion(out_fake, lab_fake)
        lossD = lossD_real + lossD_fake
        lossD.backward()
        optD.step()

        netG.zero_grad()
        lab_G = torch.full((b_size,), real_label, device=device)
        out_G = netD(fake)
        lossG = criterion(out_G, lab_G)
        lossG.backward()
        optG.step()

    # Display sample images
    with torch.no_grad():
        samples = netG(fixed_noise).cpu()
        img_grid = utils.make_grid((samples+1)/2, nrow=8)
        # Save the image to a file
        utils.save_image(img_grid, "epoch_samples.png")
        clear_output(wait=True)
        # Display the image from the file
        display(Image(filename="epoch_samples.png"))

print("Training finished.")

KeyboardInterrupt: 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
!pip install torch

In [ ]:
# Install packages
!pip install --upgrade torch torchvision # Colab usually has these already
!pip install ninja

# Clone official stylegan2-ada-pytorch
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 3.4 MB/s eta 0:00:00
Cloning into 'stylegan2-ada-pytorch'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 131 (delta 0), reused 0 (delta 0), pack-reused 129 (from 2)
Receiving objects: 100% (131/131), 1.13 MiB | 6.97 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/stylegan2-ada-pytorch


In [ ]:
# Adjust path to where you cloned or put ikea repo
# If you haven't already cloned the ikea dataset, clone it first (one-time)
!git clone https://github.com/IvonaTau/ikea.git /content/ikea_dataset || true

import os
from PIL import Image
from tqdm import tqdm

SRC_DIR = "/content/ikea_dataset/images"   # original ikea images (ImageFolder style)
OUT_DIR = "/content/ikea_resized"          # will contain resized images
TARGET_RES = 256                           # try 256 (change to 128 if OOM)
os.makedirs(OUT_DIR, exist_ok=True)

# Resize images to TARGET_RES x TARGET_RES and save to OUT_DIR
count = 0
for root, _, files in os.walk(SRC_DIR):
    for fname in files:
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            p = os.path.join(root, fname)
            try:
                img = Image.open(p).convert("RGB")
                img = img.resize((TARGET_RES, TARGET_RES), Image.LANCZOS)
                # create unique name to avoid collisions across folders
                outname = f"img_{count:06d}.png"
                img.save(os.path.join(OUT_DIR, outname))
                count += 1
            except Exception as e:
                print("skipping", p, e)
print("Saved", count, "resized images to", OUT_DIR)


Cloning into '/content/ikea_dataset'...
remote: Enumerating objects: 2407, done.
remote: Total 2407 (delta 0), reused 0 (delta 0), pack-reused 2407 (from 1)
Receiving objects: 100% (2407/2407), 86.46 MiB | 28.51 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (2540/2540), done.
Saved 2532 resized images to /content/ikea_resized


In [ ]:
# Back to stylegan2-ada-pytorch folder
%cd /content/stylegan2-ada-pytorch

# Create a .zip dataset (this will place dataset at /content/stylegan2-ada-pytorch/datasets/ikea-256.zip)
!python dataset_tool.py --source /content/ikea_resized --dest /content/stylegan2-ada-pytorch/datasets/ikea-256.zip


/content/stylegan2-ada-pytorch
  0% 0/2532 [00:00<?, ?it/s]/content/stylegan2-ada-pytorch/dataset_tool.py:429: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PIL.Image.fromarray(img, { 1: 'L', 3: 'RGB' }[channels])
100% 2532/2532 [00:19<00:00, 130.81it/s]


In [ ]:
# create a folder for pretrained nets
!mkdir -p pretrained

# Example pretrained networks hosted by NVIDIA (the training script can also fetch them directly)
# Download FFHQ pretrained (example)
!wget -c https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl -P pretrained/
# Other pretrained options: metfaces.pkl, afhqcat.pkl, artwork.pkl, etc. (if you prefer)

--2025-08-21 10:32:48--  https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl
Resolving nvlabs-fi-cdn.nvidia.com (nvlabs-fi-cdn.nvidia.com)... 3.162.174.10, 3.162.174.34, 3.162.174.74, ...
Connecting to nvlabs-fi-cdn.nvidia.com (nvlabs-fi-cdn.nvidia.com)|3.162.174.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 381624121 (364M) [binary/octet-stream]
Saving to: ‘pretrained/ffhq.pkl’

ffhq.pkl            100%[===================>] 363.94M  31.4MB/s    in 12s     

2025-08-21 10:33:01 (29.9 MB/s) - ‘pretrained/ffhq.pkl’ saved [381624121/381624121]



In [ ]:
# from inside stylegan2-ada-pytorch directory
!python train.py \
  --outdir=/content/drive/MyDrive/stylegan2_ikea_runs \
  --data=/content/stylegan2-ada-pytorch/datasets/ikea-256.zip \
  --gpus=1 \
  --cfg=stylegan2 \
  --batch=8 \
  --mirror=1 \
  --snap=10 \
  --resume=https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl


W0821 10:33:19.428000 7131 torch/utils/cpp_extension.py:118] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'

Training options:
{
  "num_gpus": 1,
  "image_snapshot_ticks": 10,
  "network_snapshot_ticks": 10,
  "metrics": [
    "fid50k_full"
  ],
  "random_seed": 0,
  "training_set_kwargs": {
    "class_name": "training.dataset.ImageFolderDataset",
    "path": "/content/stylegan2-ada-pytorch/datasets/ikea-256.zip",
    "use_labels": false,
    "max_size": 2532,
    "xflip": true,
    "resolution": 256
  },
  "data_loader_kwargs": {
    "pin_memory": true,
    "num_workers": 3,
    "prefetch_factor": 2
  },
  "G_kwargs": {
    "class_name": "training.networks.Generator",
    "z_dim": 512,
    "w_dim": 512,
    "mapping_kwargs": {
      "num_layers": 8
    },
    "synthesis_kwargs": {
      "channel_base": 32768,
      "channel_max": 512,
      "num_fp16_res": 4,
      "conv_clamp": 256
    }
  },
  "D_kwargs": {
    "class_name": "training.networks.Discriminator",
    "block

In [ ]:
# Generate 50 images using a trained network snapshot (replace path with your saved snapshot)
!python generate.py --outdir=out --trunc=0.7 --seeds=0-49 --network=/content/drive/MyDrive/stylegan2_ikea_runs/00000-stylegan2-ffhq.pkl


Loading networks from "/content/drive/MyDrive/stylegan2_ikea_runs/00000-stylegan2-ffhq.pkl"...
Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/generate.py", line 127, in <module>
    generate_images() # pylint: disable=no-value-for-parameter
    ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1442, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1363, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1226, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 794, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/decorators.py

In [ ]:
# =====================================================
# 1. Setup Environment
# =====================================================
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch

# Install dependencies
!pip install ninja opensimplex requests tqdm

# =====================================================
# 2. Download IKEA Dataset (your GitHub repo)
# =====================================================
!git clone https://github.com/IvonaTau/ikea.git
%cd ikea

# Extract dataset (if zipped) or arrange into /images folder
# Let's assume dataset is in ikea_dataset/images/*.jpg
# Move dataset to stylegan2-ada-pytorch directory
!mkdir ../datasets
!cp -r images ../datasets/ikea
%cd ..

# =====================================================
# 3. Convert dataset to StyleGAN format (tfrecords)
# =====================================================
!python dataset_tool.py --source=./datasets/ikea --dest=./datasets/ikea.zip

# =====================================================
# 4. Train StyleGAN2-ADA
# =====================================================
# --data: path to dataset zip
# --gpus: use 1 for Colab
# --batch: 32 is good for T4 GPU, adjust if memory error
# --mirror: enables random mirror augment
# --cfg=auto lets it pick best config
!python train.py \
  --outdir=./training-runs \
  --data=./datasets/ikea.zip \
  --gpus=1 \
  --batch=32 \
  --mirror=1 \
  --cfg=auto \
  --snap=5 \
  --kimg=1000

# snap=5 → saves generated samples every 5 ticks
# kimg=1000 → trains for 1000k images (reduce to 200 for faster demo)

# =====================================================
# 5. Generate Images After Training
# =====================================================
# Pick the latest network snapshot (replace with correct .pkl file)
!python generate.py \
  --outdir=./results \
  --trunc=0.7 \
  --seeds=0-50 \
  --network=./training-runs/00000-ikea-auto1/network-snapshot-000200.pkl


Cloning into 'stylegan2-ada-pytorch'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 131 (delta 0), reused 0 (delta 0), pack-reused 129 (from 2)
Receiving objects: 100% (131/131), 1.13 MiB | 7.19 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/stylegan2-ada-pytorch/stylegan2-ada-pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 4.9 MB/s eta 0:00:00
Cloning into 'ikea'...
remote: Enumerating objects: 2407, done.
remote: Total 2407 (delta 0), reused 0 (delta 0), pack-reused 2407 (from 1)
Receiving objects: 100% (2407/2407), 86.46 MiB | 25.97 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (2540/2540), done.
/content/stylegan2-ada-pytorch/stylegan2-ada-pytorch/ikea
/content/stylegan2-ada-pytorch/stylegan2-ada-pytorch
  0% 0/2532 [00:00<?, ?it/s]Error: Image width/height after scale and crop are required to be power-of-two
  0% 0/2532 [00:00

#STABLE DIFFUSER

In [ ]:
# =====================================================
# 1. Install dependencies
# =====================================================
!pip install diffusers transformers accelerate safetensors

# =====================================================
# 2. Import libraries
# =====================================================
import torch
from diffusers import StableDiffusionPipeline

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# 3. Load Stable Diffusion Model
# =====================================================
# (you can use "runwayml/stable-diffusion-v1-5" or any other)
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)
pipe = pipe.to(device)

# =====================================================
# 4. Generate IKEA furniture images
# =====================================================
prompt = "A modern IKEA wooden chair with minimalistic Scandinavian design, product photo, white background"
images = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, num_images_per_prompt=4).images

# Save images
for i, img in enumerate(images):
    img.save(f"ikea_{i}.png")


In [1]:
# =====================================================
# 1. Install dependencies
# =====================================================
!pip install diffusers transformers accelerate safetensors

# =====================================================
# 2. Import libraries
# =====================================================
import torch
from diffusers import StableDiffusionPipeline

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# 3. Load Stable Diffusion Model
# =====================================================
model_id = "runwayml/stable-diffusion-v1-5"  # You can change this to other models too

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)
pipe = pipe.to(device)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# =====================================================
# 4. Generate IKEA furniture images
# =====================================================
prompt = input("👉 Enter your prompt: ")
images = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, num_images_per_prompt=4).images

# Save images
for i, img in enumerate(images):
    img.save(f"ikea_{i}.png")

print("✅ Images saved as ikea_0.png, ikea_1.png, ikea_2.png, ikea_3.png")

#LORA TRAINING - FINE TUNING

In [18]:
# Install dependencies
!pip install diffusers==0.29.0 transformers accelerate safetensors datasets bitsandbytes
!pip install git+https://github.com/huggingface/peft.git


  Cloning https://github.com/huggingface/peft.git to /tmp/pip-req-build-3hswbl8m
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/peft.git /tmp/pip-req-build-3hswbl8m
  Resolved https://github.com/huggingface/peft.git to commit 2a27f0e00c98ea7c4ab6ecef99b13369a1cfd7eb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [19]:
import os
from datasets import load_dataset
from diffusers import StableDiffusionPipeline
from huggingface_hub import login


In [20]:
from google.colab import drive
drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/IKEA_dataset"  # change path


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from google.colab import drive
drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/IKEA_dataset"  # adjust to your path

!pip install transformers accelerate timm

from PIL import Image
import os
from transformers import BlipProcessor, BlipForConditionalGeneration

# Load BLIP model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda")

# 👇 Change this depending on where your images actually are
image_dir = dataset_path   # if images are directly inside IKEA_dataset
caption_file = dataset_path + "/captions.txt"

with open(caption_file, "w") as f:
    for img_name in os.listdir(image_dir):
        if img_name.lower().endswith((".png", ".jpg", ".jpeg")):  # only process images
            img_path = os.path.join(image_dir, img_name)
            image = Image.open(img_path).convert("RGB")

            inputs = processor(image, return_tensors="pt").to("cuda")
            out = model.generate(**inputs)
            caption = processor.decode(out[0], skip_special_tokens=True)

            f.write(f"{img_path}\t{caption}\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import pandas as pd

caption_file = dataset_path + "/captions.txt"
cleaned_file = dataset_path + "/cleaned_captions.csv"

# Load raw captions
data = []
with open(caption_file, "r") as f:
    for line in f.readlines():
        try:
            path, caption = line.strip().split("\t")
            # Clean caption (lowercase, remove unwanted chars)
            caption = caption.lower().replace(".", "").replace(",", "")
            data.append([path, caption])
        except:
            continue

# Save as CSV for training
df = pd.DataFrame(data, columns=["image_path", "caption"])
df.to_csv(cleaned_file, index=False)

print(f"✅ Cleaned captions saved to {cleaned_file}")
print(df.head())


✅ Cleaned captions saved to /content/drive/MyDrive/IKEA_dataset/cleaned_captions.csv
                                         image_path  \
0  /content/drive/MyDrive/IKEA_dataset/00000019.jpg   
1  /content/drive/MyDrive/IKEA_dataset/00000024.jpg   
2  /content/drive/MyDrive/IKEA_dataset/00000016.jpg   
3  /content/drive/MyDrive/IKEA_dataset/00000009.jpg   
4  /content/drive/MyDrive/IKEA_dataset/00000027.jpg   

                                             caption  
0    a living room with a large painting on the wall  
1            a living room with a large glass window  
2             a living room with a large white floor  
3  a room with a table and chairs and a wall with...  
4       a bedroom with a large bed and wooden floors  


In [24]:
from datasets import load_dataset, Dataset
import pandas as pd

cleaned_file = dataset_path + "/cleaned_captions.csv"

# Load cleaned captions
df = pd.read_csv(cleaned_file)

# Convert to HuggingFace dataset format
dataset = Dataset.from_pandas(df)

# Save in HuggingFace format
hf_dataset_path = dataset_path + "/hf_dataset"
dataset.save_to_disk(hf_dataset_path)

print(f"✅ HuggingFace dataset saved at {hf_dataset_path}")


Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

✅ HuggingFace dataset saved at /content/drive/MyDrive/IKEA_dataset/hf_dataset


In [29]:
import os
from PIL import Image
from torchvision import transforms
import torch

# Path to your dataset (update if needed, e.g., "/content/drive/MyDrive/IKEA_dataset")
dataset_path = "/content/drive/MyDrive/IKEA_dataset"

# Preprocessing: resize + normalize
preprocess = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Collect latents for all images
latents_list = []

for file_name in os.listdir(dataset_path):
    if file_name.endswith((".jpg", ".png", ".jpeg")):
        image_path = os.path.join(dataset_path, file_name)
        image = Image.open(image_path).convert("RGB")

        pixel_values = preprocess(image).unsqueeze(0).to("cuda")  # [1,3,512,512]

        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample() * 0.18215

        latents_list.append(latents)

print(f"Processed {len(latents_list)} images into latent space!")


Processed 30 images into latent space!


In [32]:
# 4. Encode custom text into embeddings
text = ["wooden chair", "modern sofa"]
inputs = tokenizer(
    text,
    padding="max_length",
    max_length=77,
    return_tensors="pt"
).to("cuda")

text_embeddings = text_encoder(**inputs).last_hidden_state

# 5. Match latent batch size with text batch size
batch_size = inputs["input_ids"].shape[0]
latents = torch.randn((batch_size, unet.config.in_channels, 64, 64)).to("cuda")

scheduler.set_timesteps(50)
timestep = scheduler.timesteps[0]

# 6. Run UNet forward pass
with torch.no_grad():
    model_pred_cond = unet(
        latents,
        timestep,
        encoder_hidden_states=text_embeddings
    ).sample

    model_pred_uncond = unet(
        latents,
        timestep,
        encoder_hidden_states=torch.zeros_like(text_embeddings)
    ).sample

print("✅ Conditional prediction:", model_pred_cond.shape)
print("✅ Unconditional prediction:", model_pred_uncond.shape)


RuntimeError: mat1 and mat2 must have the same dtype, but got Float and Half

In [22]:
# =====================================================
# 6. Fine-tune Stable Diffusion on IKEA dataset
# =====================================================
from diffusers import StableDiffusionPipeline, DDPMScheduler
from transformers import AutoTokenizer
from accelerate import Accelerator
from torch.utils.data import Dataset, DataLoader
import torch
from datasets import load_from_disk
import os
from PIL import Image

# Load your HuggingFace formatted dataset (from Step 5)
hf_dataset_path = dataset_path + "/hf_dataset"
dataset = load_from_disk(hf_dataset_path)

# Tokenizer for text captions
tokenizer = AutoTokenizer.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="tokenizer")

# Custom PyTorch dataset wrapper
class IKEADataset(Dataset):
    def __init__(self, dataset, tokenizer, size=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.size = size

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        # Load image
        image = Image.open(item["image_path"]).convert("RGB")
        image = image.resize((self.size, self.size))

        # Convert image to tensor
        image = torch.tensor(list(image.getdata()), dtype=torch.float32).reshape(self.size, self.size, 3)
        image = image.permute(2,0,1) / 255.0   # [C,H,W] normalized 0-1

        # Tokenize caption
        caption = item["caption"]
        inputs = self.tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        )

        return {
            "pixel_values": image,
            "input_ids": inputs.input_ids.squeeze(0)
        }

# Prepare dataloader
train_dataset = IKEADataset(dataset, tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# Load Stable Diffusion UNet + Scheduler
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    "runwayml/stable-diffusion-v1-5", subfolder="unet"
)

optimizer = torch.optim.AdamW(unet.parameters(), lr=5e-6)

# Accelerator for multi-GPU or CPU fallback
accelerator = Accelerator()
unet, optimizer, train_dataloader = accelerator.prepare(unet, optimizer, train_dataloader)

# Training loop
epochs = 1   # start small (increase later)
unet.train()

for epoch in range(epochs):
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()

        pixel_values = batch["pixel_values"].to(accelerator.device)
        input_ids = batch["input_ids"].to(accelerator.device)

        # Forward pass (dummy noise schedule)
        noise = torch.randn_like(pixel_values)
        timesteps = torch.randint(0, 1000, (pixel_values.shape[0],), device=pixel_values.device).long()

        # Predict noise
        model_pred = unet(pixel_values, timesteps, encoder_hidden_states=None).sample

        loss = torch.nn.functional.mse_loss(model_pred, noise)

        accelerator.backward(loss)
        optimizer.step()

        if step % 10 == 0:
            print(f"Epoch {epoch}, Step {step}, Loss: {loss.item()}")

# Save fine-tuned UNet
save_path = "./ikea_unet"
accelerator.unwrap_model(unet).save_pretrained(save_path)
print(f"✅ Fine-tuned model saved at {save_path}")


RuntimeError: Given groups=1, weight of size [320, 4, 3, 3], expected input[2, 3, 512, 512] to have 4 channels, but got 3 channels instead